In [ ]:
import nest_asyncio
nest_asyncio.apply()

In [ ]:
## 돌발과제 !
# data폴더 안 html만 노드파서
# data폴더 안 마크다운(md)만 노드파서

In [ ]:
from llama_index.core.node_parser import HTMLNodeParser
from llama_index.core import SimpleDirectoryReader

# html파일 불러오기
html_docs = SimpleDirectoryReader(input_dir='./data', required_exts=['.html']).load_data()

# html노드파서 사용
parser = HTMLNodeParser()
html_nodes = parser.get_nodes_from_documents(html_docs)

In [ ]:
# html 노드 텍스트
html_nodes[0].text

In [ ]:
# 첫번째 노드 메타데이터
html_nodes[0].metadata

In [ ]:
# 마크다운(.md) 노드 파서 
from llama_index.core.node_parser import MarkdownNodeParser
from llama_index.core import SimpleDirectoryReader

# 마크다운 파일 로드
markdown_docs = SimpleDirectoryReader(input_dir='./data', required_exts=['.md']).load_data()

parser = MarkdownNodeParser()
markdown_nodes = parser.get_nodes_from_documents(markdown_docs)


In [ ]:
len(markdown_nodes)

In [ ]:
markdown_nodes[0].text

In [ ]:
markdown_nodes[0].metadata

# 벡터 스토어

- 임베딩을 통해 생성된 고차원 백터 데이터를 효율적으로 저장하고 검색할 수 있도록 설계된 데이터베이스
- 벡터 간의 거리(유클리드 거리, 코사인 유사도 등)나 유사도를 기반으로 데이터를 검색한다.
- 자연어 처리 (NLP), 추천 시스템, 이미지 검색 비정형 데이터를 다루는 ai 앱에서 중요한 역할을 한다.


## FAISS

- 메타에서 만든 고성능 벡터 검색 라이브러리
- 대규모 데이터세트에서도 빠르게 유사한 벡터를 검색할 수 있는 도구
- 최근접 이웃 탐색(NNS)에 최적화 되어 있다.
- GPU 가속 기능을 통해 초고속 검색 성능을 제공
- 메모리에서 동작하므로 매우 빠르다.
    - 데이터를 저장하지 않고, 프로그램 종료 시 메모리에 있는 데이터가 사라지므로 
    - 별도의 데이터를 저장하거나 복원하는 작업이 필요하다.
- 로컬 환경에서 사용
- 클라우드 기반의 확장성을 지원하지 않는다.


In [ ]:
import faiss
from llama_index.vector_stores.faiss import FaissVectorStore
from llama_index.core import StorageContext
from llama_index.llms.openai import OpenAI
from llama_index.core import Settings
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')

In [ ]:
# llm 설정
Settings.llm = OpenAI(model='gpt-4o', temperature=0.5, max_tokens=200, context_window=4096)

In [ ]:
# 데이터 로드
documents = SimpleDirectoryReader('./data/pdf_sample2').load_data()

In [ ]:
# FAISS 인덱스 생성
dimension = 1536 # OpenAI 임베딩 차원
faiss_index = faiss.IndexFlatL2(dimension)

In [ ]:
# FAISS를 라마인덱스의 인덱싱 및 검색 파이프라인에 통합
vector_store = FaissVectorStore(faiss_index)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

# 문서 내용을 벡터 DB에 저장
index = VectorStoreIndex.from_documents(documents, storage_context=storage_context)

In [ ]:
# FAISS 인덱스 저장
faiss.write_index(faiss_index, 'faiss.index')

In [ ]:
# FAISS 인덱스 로드
loaded_index = faiss.read_index('faiss.index')

In [ ]:
# 쿼리 엔진 생성
query_engine = index.as_query_engine()

# 쿼리 실행
query = '이 논문에서 제안하는 모델의 장점은 무엇인가요?'
response = query_engine.query(query)

In [ ]:
# 응답 출력
print(f'질문 --> {query}')
print()
print(f'답변 -> {response}')

## ChromaDB

- 간편하고 실용적인 벡터 데이터베이스
- 임베딩 데이터를 저장하고 유사한 데이터를 검색할 수 있는 기능을 제공
- 자동 데이터 저장 및 지속성 기능을 통해 데이터 쉽게 관리
- 메타 데이터 관리 및 필터링 같은 고급 기능도 지원
- 파이썬 중심의 개발자 친화적인 api제공
- 설치와 사용이 간단 --> 중소 규모 프로젝트나 프로토타입
- 시각화 도구를 통해 저장하므로 직관적, 데이터 관리가 쉽다.
- 클라우드와 로컬 환경 모두에서 사용가능한 유연성이 높다.

In [ ]:
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore

In [ ]:
Settings.llm

In [ ]:
documents

In [ ]:
# 벡터 DB 생성 및 저장
db = chromadb.PersistentClient(path='./data/chroma_db') # 저장할 경로 지정

# 컬렉션 (관련된 데이터를 모아두는 디지털 서랍장)
chroma_collection = db.get_or_create_collection('quickstart')

In [ ]:
# 크로마db를 라마인덱스의 인덱싱 및 검색 파이프라인에 통합
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

In [ ]:
# 쿼리 엔진 생성
query_engine = index.as_query_engine()

# 쿼리 실행
query = '이 논문에서 제안하는 모델의 특징은 무엇인가요?'
response = query_engine.query(query)

In [ ]:
print(f'질문 --> {query}')
print()
print(f'답변 --> {response}')

## 기존 벡터 DB 불러오기

In [ ]:
# 클라이언트 초기화
db = chromadb.PersistentClient(path='./data/chroma_db')

# 컬렉션 가져오기
chroma_collection = db.get_or_create_collection('quickstart')

vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

In [ ]:
# 지정된 벡터로부터 인덱스 로드
index = VectorStoreIndex.from_vector_store(vector_store, storage_context=storage_context)

In [ ]:
# 쿼리 엔진 생성
query_engine = index.as_query_engine()

query = '이 논문에서 제안하는 모델의 특징은 무엇인가요?'
response = query_engine.query(query)

In [ ]:
print(response)

## 목적--> 질문을 통해 문서 혹은 자료에서 우리가 궁금해하는 내용을 손쉽게 파악

## 문서를 텍스트 직접 입력으로 생성하고 인덱스도 생성

In [ ]:
from llama_index.core import Document 
from llama_index.core import VectorStoreIndex

# 문서 생성
documents = [
    Document(text='라마인덱스는 강력한 검색 엔진을 구축할 수 있는 오픈소스 라이브러리입니다.'),
    Document(text='라마인덱스는 다양한 검색 방법을 지원합니다. (벡터 검색, 키워드 검색)'),
    Document(text='벡터 검색은 문서의 의미적 유사도를 기반으로 정보를 검색하는 방법입니다.'),
    Document(text='키워드 검색은 특정 단어가 포함된 문서를 검색하는 방식입니다.'),
    Document(text='이 라이브러리는 python기반이며, 빠르고 유연한 검색 기능을 제공합니다.')
]

# Document에서 바로 인덱스 생성
index = VectorStoreIndex.from_documents(documents)

- index.as_query_engine()과 같은 방식으로 검색하고 답변을 받았다.
- 검색기 --> index.as_retriever() 만 설정(자동으로 검색기의 종류가 정해지는 메서드)

In [ ]:
# 벡터 검색
retriever = index.as_retriever(
    similarity_top_k=1, # 상위 1개 문서 검색
    search_type='vector' # 기본값은 'vector'
)

In [ ]:
# 질문에 대한 맥락 검색

# RetrieverQueryEngine

- 고급 사용자를 위한 세부적인 설정이 가능하다.
- 기본 검색기를 교체하거나, 여러 개의 후처리기를 적용 하는 등 세부 설정 가능

In [ ]:
from llama_index.core import get_response_synthesizer
from llama_index.core.postprocessor import SimilarityPostprocessor
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.node_parser import SentenceSplitter

In [ ]:
# 문서 로드 및 인덱스 생성
documents = SimpleDirectoryReader(input_files=['./data/txt_example.txt']).load_data()

In [ ]:
# 노드 파서 설정 수정
node_parser = SentenceSplitter(
    chunk_size=100,
    chunk_overlap=20,
    separator='₩n' # 엔터(빈 줄)을 기준으로 분할
)

In [ ]:
# 문서 분할
nodes = node_parser.get_nodes_from_documents(documents)

# 총 노드 수 확인
print(len(nodes))

In [ ]:
# 검색기 생성 (벡터 유사도 검색)
retriever = index.as_retriever(similarity_top_k=5)

In [ ]:
# 후처리기 생성 (유사도 0.7이상만 필터링)
# 너무 값을 높게 설정하면 오히려 아무 노드도 답변에 참고하지
# 못하고 훨씬 엉뚱한 답변이 돌아온다. 주의! 
postprocessor = SimilarityPostprocessor(similarity_cutoff=0.7)

In [ ]:
# 응답 생성기 생성
# "compact" --> 아무 설정도 하지 않았을 때 입력되는 기본값 
#           --> 세부 정보가 부족
# "refine"  --> 단계적으로 응답을 계선하는 방식 -> 비용이 많이 든다. 속도가 느리다.
# "tree_summarize" --> 트리 구조 활용, 검색된 문서의 내용들을 계층적으로 요약, 
#                 --> 검색된 문서가 많을 때 전체적인 개요를 파악하기 좋다.
response_synthesizer = get_response_synthesizer(response_mode='compact')

In [ ]:
# 쿼리 엔진 생성
query_engine = RetrieverQueryEngine(
    retriever=retriever,
    response_synthesizer=response_synthesizer,
    node_postprocessors=[postprocessor]
)

In [ ]:
# 쿼리 실행
response = query_engine.query('인공지능에 대해서 요약해줘.')

In [ ]:
response

# Chat Engine

- 이전 대화를 기억해서 답변을 받고자 할 때 사용
- as_chat_engine()함수 사용

In [ ]:
documents = SimpleDirectoryReader(input_files=['./data/pdf_example.pdf']).load_data()
index = VectorStoreIndex.from_documents(documents)

In [ ]:
chat_engine = index.as_chat_engine()

In [ ]:
response = chat_engine.chat('rag가 뭐야?')
print(response)